# Exercise 5: Real-World Analysis Project

**Scenario:**
You are a Data Analyst at AdventureWorks. Management needs a report on business performance.

**Your Tasks:**
1. Load data and check quality
2. Calculate KPIs (Revenue, Products, Regions, Customers)
3. Create visualizations
4. Export results
5. Write a management summary

**Estimated Time:** 60-90 minutes

**Prerequisites:** Exercise 1-4 completed

In [ ]:
# Idempotent notebook-dep installer (safe to rerun).
# Uses uv from the Codespace; falls back to pip if running standalone.
import importlib.util, subprocess, sys, shutil
_pkgs = {'sql': 'jupysql', 'duckdb': 'duckdb',
         'duckdb_engine': 'duckdb-engine', 'prettytable': 'prettytable'}
_missing = [p for m, p in _pkgs.items() if importlib.util.find_spec(m) is None]
if _missing:
    cmd = (['uv', 'pip', 'install', '--python', sys.executable]
           if shutil.which('uv')
           else [sys.executable, '-m', 'pip', 'install'])
    subprocess.check_call(cmd + _missing)
    print('Installed:', _missing)
else:
    print('All notebook deps already present.')


---
## Part 1: Load Data

Load all required tables from `../sample_data/AW_CSV/`:

| File | Table Name | Description |
|------|------------|-------------|
| Production.Product.csv | products | Products |
| Production.ProductCategory.csv | categories | Categories |
| Production.ProductSubcategory.csv | subcategories | Subcategories |
| Sales.SalesOrderHeader.csv | orders | Order headers |
| Sales.SalesOrderDetail.csv | order_details | Order line items |
| Sales.SalesTerritory.csv | territories | Sales regions |
| Sales.Customer.csv | customers | Customers |

**Syntax:**
```sql
CREATE TABLE tablename AS 
SELECT * FROM read_csv_auto('../sample_data/AW_CSV/filename.csv')
```

In [ ]:
# Your code here:
# Load all 7 tables








print('✓ All tables loaded')

---
## Part 2: Check Data Quality

### 2a) Table Overview

Show for each table:
- Row count
- Column count

**Expected Format:**
```
products: X rows, Y columns
categories: X rows, Y columns
...
```

In [ ]:
# Your code here:
tables = ['products', 'categories', 'subcategories', 'orders', 'order_details', 'territories', 'customers']

for table in tables:
    # Count rows and columns
    pass

### 2b) Check for NULL values

Check the `orders` table for NULL values in important columns:
- `CustomerID`
- `TotalDue`
- `OrderDate`

**Syntax:**
```sql
SELECT 
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer,
    SUM(CASE WHEN TotalDue IS NULL THEN 1 ELSE 0 END) AS null_totaldue,
    SUM(CASE WHEN OrderDate IS NULL THEN 1 ELSE 0 END) AS null_orderdate
FROM orders
```

In [ ]:
# Your code here:


---
## Part 3: KPI 1 - Revenue Overview

### 3.1 Total Revenue

Calculate the **total revenue** from the `orders` table (column: `TotalDue`).

**Expected Result:** A single number, formatted as currency (e.g., $109,846,381.40)

In [ ]:
# Your code here:


### 3.2 Monthly Revenue with Trend

Calculate **monthly revenue** with:
- Year-Month
- Revenue for the month
- Change vs. previous month (absolute and percentage)

**Columns:** month, revenue, prev_month_revenue, change_abs, change_pct

**Tips:**
- `strftime(OrderDate, '%Y-%m')` for year-month
- `LAG(revenue) OVER (ORDER BY month)` for previous month

In [ ]:
# Your code here:


### 3.3 Visualization: Revenue Trend

Create a **line chart** with monthly revenue.

**Tips:**
```python
df = conn.execute("SELECT ...").df()
fig = px.line(df, x='month', y='revenue', title='Monthly Revenue')
fig.show()
```

In [ ]:
# Your code here:


---
## Part 4: KPI 2 - Product Analysis

### 4.1 Top 10 Products by Revenue

Show the **10 highest revenue products**.

**Columns:** product_name, total_revenue, total_quantity

**Tables:** order_details JOIN products
**JOIN Column:** ProductID
**Revenue Column:** LineTotal

In [ ]:
# Your code here:


### 4.2 Category Analysis

Analyze each **product category**:

**Columns:**
- category_name
- product_count (number of products)
- total_revenue
- avg_price (average price)
- revenue_share (% of total revenue)

**Table Chain:** order_details → products → subcategories → categories

In [ ]:
# Your code here:


### 4.3 Visualization: Categories

Create a **pie chart** with revenue share per category.

**Tips:**
```python
fig = px.pie(df, values='total_revenue', names='category_name', title='Revenue by Category')
fig.show()
```

In [ ]:
# Your code here:


---
## Part 5: KPI 3 - Regional Analysis

### 5.1 Revenue per Region

Show **revenue per territory**.

**Columns:**
- territory_name
- order_count
- total_revenue
- avg_order_value

**Tables:** orders JOIN territories
**JOIN Column:** TerritoryID

In [ ]:
# Your code here:


### 5.2 Visualization: Regions

Create a **horizontal bar chart** with revenue per region.

**Tips:**
```python
fig = px.bar(df, x='total_revenue', y='territory_name', orientation='h', 
             title='Revenue by Territory')
fig.show()
```

In [ ]:
# Your code here:


---
## Part 6: KPI 4 - Customer Analysis

### 6.1 Average Order Value

Calculate the **average order value** (TotalDue).

**Expected Result:** A single number (approximately $3,500)

In [ ]:
# Your code here:


### 6.2 Customer Distribution by Order Count

How many customers have 1, 2, 3, ... orders?

**Columns:** order_count, customer_count

**Tips:**
1. First count orders per customer (subquery)
2. Then group by that count

In [ ]:
# Your code here:


### 6.3 Pareto Analysis (80/20 Rule)

**Question:** What percentage of revenue do the top 20% of customers generate?

**Steps:**
1. Calculate revenue per customer
2. Sort by revenue descending
3. Calculate cumulative percentage
4. Find the point where 20% of customers is reached

**Syntax:**
```sql
WITH customer_revenue AS (
    SELECT CustomerID, SUM(TotalDue) AS revenue
    FROM orders
    GROUP BY CustomerID
),
ranked AS (
    SELECT 
        CustomerID,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rank,
        SUM(revenue) OVER (ORDER BY revenue DESC) AS cumulative_revenue,
        SUM(revenue) OVER () AS total_revenue
    FROM customer_revenue
)
SELECT 
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_revenue), 1) AS customer_pct,
    ROUND(100.0 * MAX(cumulative_revenue) / MAX(total_revenue), 1) AS revenue_pct
FROM ranked
WHERE rank <= (SELECT COUNT(*) * 0.2 FROM customer_revenue)
```

In [ ]:
# Your code here:


---
## Part 7: Executive Dashboard

Create a **dashboard** with 4 charts in a grid:

| Position | Chart |
|----------|-------|
| Top left | Monthly Revenue (Line) |
| Top right | Revenue by Category (Pie) |
| Bottom left | Revenue by Region (Bar) |
| Bottom right | Top 5 Products (Bar) |

**Syntax:**
```python
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scatter"}, {"type": "pie"}],
           [{"type": "bar"}, {"type": "bar"}]],
    subplot_titles=('Monthly Revenue', 'Revenue by Category', 
                    'Revenue by Region', 'Top 5 Products')
)

# Chart 1: Line
fig.add_trace(go.Scatter(x=df1['month'], y=df1['revenue'], mode='lines'), row=1, col=1)

# Chart 2: Pie
fig.add_trace(go.Pie(values=df2['revenue'], labels=df2['category']), row=1, col=2)

# Chart 3 & 4: Bars
fig.add_trace(go.Bar(x=df3['revenue'], y=df3['territory'], orientation='h'), row=2, col=1)
fig.add_trace(go.Bar(x=df4['product'], y=df4['revenue']), row=2, col=2)

fig.update_layout(height=800, title_text="AdventureWorks Executive Dashboard")
fig.show()
```

In [ ]:
# Your code here:
# 1. Load the 4 DataFrames
# 2. Create the dashboard


---
## Part 8: Export Data

Export the most important results:

| File | Content | Format |
|------|---------|--------|
| monthly_revenue.csv | Monthly revenues | CSV |
| category_analysis.parquet | Category analysis | Parquet |
| top_products.json | Top 10 products | JSON |

**Syntax:**
```sql
COPY (SELECT ...) TO '../exports/file.csv' (HEADER, DELIMITER ',')
COPY (SELECT ...) TO '../exports/file.parquet' (FORMAT PARQUET)
COPY (SELECT ...) TO '../exports/file.json' (FORMAT JSON, ARRAY true)
```

In [ ]:
# Your code here:



print('✓ Data exported to ../exports/')

---
## Part 9: Management Summary

Summarize your findings. Edit the markdown cell below:

### 📊 Management Summary: AdventureWorks Analysis

**Analysis Period:** [Enter period]

---

#### Top 3 Insights:

1. **[Insight 1]**
   - Details...

2. **[Insight 2]**
   - Details...

3. **[Insight 3]**
   - Details...

---

#### Top 3 Recommendations:

1. **[Recommendation 1]**
   - Rationale...

2. **[Recommendation 2]**
   - Rationale...

3. **[Recommendation 3]**
   - Rationale...

---

#### Next Steps:

- [ ] [Step 1]
- [ ] [Step 2]
- [ ] [Step 3]

---
## 🎉 Project Complete!

You've completed a full analysis project with DuckDB!

**What you learned:**
- ✅ Load data and check quality
- ✅ Calculate KPIs with complex queries
- ✅ Apply Window Functions and CTEs
- ✅ Create interactive visualizations
- ✅ Export results in various formats
- ✅ Communicate business insights

**Further Ideas:**
- Predictive Analytics (revenue forecasting)
- Customer Lifetime Value (CLV)
- Churn Analysis
- A/B Test Evaluation